# Aims
- to implement the QJL part, i need to dequantise and unrotate the embeddings
- This is so that I can calculate the residuals and also the QJL signs

In [2]:
import numpy as np
import polars as pl
import pickle as pkl
import sys
sys.path.append("..")
from lib.turboquant.turboquant import dequantize_embeddings_polar, calculate_centroids_lookup_table

In [2]:
with open("../data/codebook/384d_codebook.pkl", "rb") as f:
    codebook = pkl.load(f)

In [3]:
data = pl.scan_parquet("../data/processed/arxiv_quantized_embeddings_4bits.parquet").head(3000).collect()

In [4]:
data

id,packed_embedding
str,"array[u8, 256]"
"""0704.0001""","[101, 150, … 118]"
"""0704.0002""","[40, 199, … 173]"
"""0704.0003""","[162, 75, … 102]"
"""0704.0004""","[33, 124, … 68]"
"""0704.0005""","[125, 47, … 100]"
…,…
"""0704.3046""","[123, 58, … 104]"
"""0704.3047""","[201, 70, … 131]"
"""0704.3048""","[83, 158, … 168]"


In [5]:
data = data.with_columns(
    pl.col("packed_embedding").map_batches(lambda s: dequantize_embeddings_polar(s, codebook, 4, 42, 384), return_dtype=pl.Array(pl.Float32, 384)).alias("unrot")
)

In [6]:
data.head()

id,packed_embedding,unrot
str,"array[u8, 256]","array[f32, 384]"
"""0704.0001""","[101, 150, … 118]","[-0.160321, -0.006887, … 0.048833]"
"""0704.0002""","[40, 199, … 173]","[0.00625, 0.043267, … 0.020498]"
"""0704.0003""","[162, 75, … 102]","[0.019474, -0.033363, … 0.010709]"
"""0704.0004""","[33, 124, … 68]","[-0.069002, -0.021498, … -0.00157]"
"""0704.0005""","[125, 47, … 100]","[0.037676, 0.00342, … -0.030098]"


In [ ]:
d.reshape((-1, 2))

array([[ 1,  2],
       [ 3,  4],
       [ 5,  6],
       [ 7,  8],
       [ 9, 10],
       [11, 12]])

In [ ]:
d.reshape((-1, 2))[np.array([[2,3,4], [4,4, 5]])].reshape(2, -1)

array([[ 5,  6,  7,  8,  9, 10],
       [ 9, 10,  9, 10, 11, 12]])

In [ ]:
d.reshape((-1, 2))[np.array([[2,3,4], [4,4, 5]])].shape

(2, 3, 2)

In [4]:
e = pl.read_parquet("../data/processed/arxiv_quantized_embeddings_4bits_qjl.parquet")

In [8]:
e["packed_qjl_bits"].to_numpy().sum()

np.uint64(0)

In [9]:
e

id,packed_embedding,gamma,packed_qjl_bits
str,"array[u8, 256]",f32,"array[u8, 64]"
"""0704.0001""","[31, 31, … 31]",2.813821,"[0, 0, … 0]"
"""0704.0002""","[31, 31, … 31]",2.897824,"[0, 0, … 0]"
"""0704.0003""","[31, 31, … 31]",2.954786,"[0, 0, … 0]"
"""0704.0004""","[31, 31, … 31]",2.879711,"[0, 0, … 0]"
"""0704.0005""","[31, 31, … 31]",2.934319,"[0, 0, … 0]"
…,…,…,…
"""0705.1067""","[31, 31, … 31]",2.928838,"[0, 0, … 0]"
"""0705.1068""","[31, 31, … 31]",2.89887,"[0, 0, … 0]"
"""0705.1069""","[31, 31, … 31]",2.909322,"[0, 0, … 0]"
